# Exercise — Choose a Governance Model for Trailhead Provisions

**Trailhead Provisions** grew by acquisition: four domain teams — **E-commerce Orders** (led by the **E-commerce Orders Lead**), **Customer Identity & Loyalty** (led by the **Customer Identity Lead**), **Inventory & Fulfillment** (led by the **Inventory & Fulfillment Lead**), and **Marketing & Campaigns** (led by the **Marketing & Campaigns Lead**) — own their pipelines and definitions on a shared AWS platform. Alongside them sit three cross-cutting roles: a **Central Governance Lead**, a **Platform Engineering Lead** (leading the Platform Engineering team, which manages pipeline infrastructure & OpenLineage instrumentation), and a Finance-appointed **CSRD Reporting Owner** (owns sustainability disclosure definitions).

A **GDPR audit** and a new **CSRD sustainability mandate** now create obligations that cross every domain. Decide how to govern. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

# Auto-load .env from the workspace or parent directories if present
from pathlib import Path
for _parent in (Path.cwd(), *Path.cwd().parents):
    _env_file = _parent / ".env"
    if _env_file.is_file():
        for _raw in _env_file.read_text(encoding="utf-8").splitlines():
            _line = _raw.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                _k, _v = _k.strip(), _v.strip().strip("\"").strip("'")
                if _k and _v and not os.environ.get(_k):
                    os.environ[_k] = _v
        break

import boto3
from botocore.exceptions import ClientError, NoCredentialsError
import setup_aws

_CRED_MSG = (
    "AWS credentials are missing or expired.\n"
    "1. Open Cloud Resources and start your Cloud Lab.\n"
    "2. Copy AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, and AWS_SESSION_TOKEN.\n"
    "3. Uncomment and paste them into Step 0 above, then re-run Step 0 (and re-run in each new lab session)."
)

# Verify credentials before seeding
sts = boto3.client("sts", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    sts.get_caller_identity()
except NoCredentialsError:
    raise RuntimeError(_CRED_MSG) from None
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code", "")
    if code in {"ExpiredToken", "ExpiredTokenException", "InvalidClientTokenId", "UnrecognizedClientException", "AuthFailure", "AccessDeniedException"}:
        raise RuntimeError(_CRED_MSG) from None
    raise
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
pd.DataFrame([{"domain": gc.catalog.table(t).domain, "table": t,
               "owner": gc.catalog.table(t).owner or "(no owner set)"}
              for t in gc.tables()]).sort_values("domain").reset_index(drop=True)

## 1a. Governance-model decision
Choose **centralized**, **decentralized**, or **federated**, and state which decisions stay central and which side each role sits on.

> **Your governance-model decision here.** Address:
>
> - Which model (centralized / decentralized / federated), justified from Trailhead's domain autonomy AND its GDPR/CSRD exposure.
> - Which side each role sits on (**Central Governance Lead**, **CSRD Reporting Owner**, and **Platform Engineering Lead** vs. the four domain leads: **E-commerce Orders Lead**, **Customer Identity Lead**, **Inventory & Fulfillment Lead**, **Marketing & Campaigns Lead**).
> - Which specific decisions stay CENTRAL even within your model, and why each must.
> - Which decisions remain with the domains.

## 1b. RACI (governance functions × domains)
One single individual **Accountable (`A`)** owner per function; name the enforcement authority and escalation path.

> **Your RACI here.** Cover all 4 domains × the governance functions
> (PII classification & tagging, consent system of record, data contract standard, data quality SLO policy, lineage instrumentation, carbon metric definition, access policy), with **exactly ONE single Accountable (`A`) individual** per function, plus the enforcement authority and an escalation path.

## 1c. Governance topology
Draw the central core, the four domains, and the key cross-domain flows as a Mermaid diagram.

> **Your Mermaid topology here** — show the central core, the four
> domains, and the key cross-domain data flows (consent, order→carbon→report).